In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import os

from params import Config
from utils import SetSeeds
from net import net
from dataset import DeapDataset, MultimodalDataset
from Dataloader_Excel import read_excel_eeg, read_excel_nirs
from train_validate import train
from loss_function import MySpl_loss
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold

if __name__ == '__main__':
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    if not os.path.exists('Results'):
        os.mkdir('Results')

    for subject in range(29):
        subject += 1
        seed = np.random.randint(1, 2025)
        print('seed is {}'.format(seed))
        print('training on:', device)
        SetSeeds(seed)

        args = Config().parse()

        mode = 0
        eeg, labels = read_excel_eeg(subject, mode)
        nirs, _ = read_excel_nirs(subject, mode)
        dataset = MultimodalDataset(eeg, nirs, labels)
        # dataset = DeapDataset(path_view1 = args.path_eeg, path_view2 = args.path_face, path_label = args.path_label)

        K = 5
        KF = KFold(n_splits=K, shuffle=False) # 80/20 for training/testing

        acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []

        fold = 1
        for train_idx, test_idx in KF.split(dataset):
            train_subsampler = torch.utils.data.SubsetRandomSampler(train_idx)
            val_subsampler = torch.utils.data.SubsetRandomSampler(test_idx)
            train_loader = DataLoader(dataset, batch_size=args.batch_size, sampler=train_subsampler)
            eval_loader = DataLoader(dataset, batch_size=args.batch_size, sampler=val_subsampler)
        
            model = net(eeg_dim=4000, nirs_dim=200, 
                        hidden_size=256, num_layers=1, 
                        dim=256, heads=1, dim_head=256, 
                        mlp_dim=512, num_classes=2, dropout=0.5).to(device, dtype=torch.float64)
            
            optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=1e-5)
            scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
            
            criterion1 = nn.CrossEntropyLoss()
            criterion2 = MySpl_loss(n_samples=len(train_loader.dataset), batch_size=args.batch_size, alpha=args.alpha, beta=args.beta, spl_lambda=args.spl_lambda, spl_gamma=args.spl_gamma)
            criterions = [criterion1, criterion2]

            _, _, acc, precision, recall, f1, kappa, _, _ = train(model, criterions, optimizer, scheduler, train_loader, eval_loader, device, args.epochs, args.alpha, args.beta, fold, args.task)

            acc_list.extend(acc)
            precision_list.extend(precision)
            recall_list.extend(recall)
            f1_list.extend(f1)
            kappa_list.extend(kappa)

            fold += 1
        
        mean_acc, std_acc = np.mean(acc_list), np.std(acc_list)
        mean_precision, std_precision = np.mean(precision_list), np.std(precision_list)
        mean_recall, std_recall = np.mean(recall_list), np.std(recall_list)
        mean_f1, std_f1 = np.mean(f1_list), np.std(f1_list)
        mean_kappa = np.mean(kappa_list)

        mean_acc, std_acc = mean_acc * 100, std_acc * 100
        mean_precision, std_precision = mean_precision * 100, std_precision * 100
        mean_recall, std_recall = mean_recall * 100, std_recall * 100
        mean_f1, std_f1 = mean_f1 * 100, std_f1 * 100

        log_path = f'Results/log_s{subject:2d}.txt'
        with open(log_path, 'a') as log_file:
            log_file.write(f'Accuracy: {mean_acc:.2f} ± {std_acc:.2f}\n')
            log_file.write(f'Precision: {mean_precision:.2f} ± {std_precision:.2f}\n')
            log_file.write(f'Recall: {mean_recall:.2f} ± {std_recall:.2f}\n')
            log_file.write(f'F1: {mean_f1:.2f} ± {std_f1:2f}\n')
            log_file.write(f'Kappa: {mean_kappa:.2f}\n')
            log_file.write('\n')

        log_excel = f'Results/Log.xlsx'
        new_row = pd.DataFrame([[mean_acc, std_acc, mean_precision, std_precision, mean_recall, std_recall, mean_f1, std_f1, mean_kappa]], 
                            columns=['Accuracy', 'Std_Accuracy', 'Precision', 'Std_Precision', 'Recall', 'Std_Recall', 'F1', 'Std_F1', 'Kappa'])
        new_row = new_row.round(2)
        if not os.path.exists(log_excel):
            new_row.to_excel(log_excel, index=False, engine='openpyxl')
        else:
            with pd.ExcelWriter(log_excel, mode='a', engine='openpyxl', if_sheet_exists='overlay') as writer:
                if 'Sheet1' in writer.sheets:
                    startrow = writer.sheets['Sheet1'].max_row
                else:
                    startrow = 0
                new_row.to_excel(writer, index=False, header=False, startrow=startrow)

        print(f'Subject:{subject} Accuracy: {mean_acc:.2f} ± {std_acc:.2f}\n')

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


seed is 720
training on: cuda:0
1 EEG torch.Size([60, 30, 4000]) Labels torch.Size([60])
1 fNIRS torch.Size([60, 36, 200]) Labels torch.Size([60])


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


< Fold1 0% 1/200 1.238s >train_loss = 0.1393 train_acc = 0.5625 valid_loss = 36.5139 valid_acc = 0.5000
< Fold1 1% 2/200 0.199s >train_loss = 0.0000 train_acc = 0.5000 valid_loss = 46.9965 valid_acc = 0.5000


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

...

< Fold2 25% 50/200 0.241s >train_loss = 0.6475 train_acc = 0.7083 valid_loss = 0.6796 valid_acc = 0.5000


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


< Fold2 26% 51/200 0.267s >train_loss = 0.7483 train_acc = 0.5417 valid_loss = 0.6804 valid_acc = 0.5000
< Fold2 26% 52/200 0.271s >train_loss = 0.6984 train_acc = 0.6042 valid_loss = 0.6705 valid_acc = 0.5833
< Fold2 26% 53/200 0.253s >train_loss = 0.6628 train_acc = 0.6250 valid_loss = 0.6670 valid_acc = 0.5833
< Fold2 27% 54/200 0.251s >train_loss = 0.7013 train_acc = 0.5625 valid_loss = 0.6501 valid_acc = 0.7500
< Fold2 28% 55/200 0.255s >train_loss = 0.6700 train_acc = 0.6667 valid_loss = 0.6320 valid_acc = 0.9167
< Fold2 28% 56/200 0.252s >train_loss = 0.6892 train_acc = 0.6875 valid_loss = 0.6058 valid_acc = 0.8333
< Fold2 28% 57/200 0.244s >train_loss = 0.6528 train_acc = 0.6667 valid_loss = 0.6937 valid_acc = 0.5833
< Fold2 29% 58/200 0.279s >train_loss = 0.7114 train_acc = 0.6042 valid_loss = 0.6139 valid_acc = 0.8333
< Fold2 30% 59/200 0.251s >train_loss = 0.6491 train_acc = 0.6875 valid_loss = 0.6068 valid_acc = 0.7500


/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


< Fold2 30% 60/200 0.239s >train_loss = 0.5882 train_acc = 0.7500 valid_loss = 0.7293 valid_acc = 0.5000
< Fold2 30% 61/200 0.255s >train_loss = 0.5161 train_acc = 0.8125 valid_loss = 0.5635 valid_acc = 0.7500
< Fold2 31% 62/200 0.254s >train_loss = 0.5550 train_acc = 0.7083 valid_loss = 0.6622 valid_acc = 0.7500
< Fold2 32% 63/200 0.238s >train_loss = 0.4455 train_acc = 0.8750 valid_loss = 0.7590 valid_acc = 0.7500
< Fold2 32% 64/200 0.244s >train_loss = 0.4622 train_acc = 0.8542 valid_loss = 0.5592 valid_acc = 0.7500
< Fold2 32% 65/200 0.244s >train_loss = 0.3672 train_acc = 0.8958 valid_loss = 0.5862 valid_acc = 0.7500
< Fold2 33% 66/200 0.243s >train_loss = 0.3820 train_acc = 0.8750 valid_loss = 0.5373 valid_acc = 0.9167
< Fold2 34% 67/200 0.242s >train_loss = 0.3327 train_acc = 0.9167 valid_loss = 0.7106 valid_acc = 0.8333
< Fold2 34% 68/200 0.245s >train_loss = 0.3156 train_acc = 0.9167 valid_loss = 0.5154 valid_acc = 0.8333
< Fold2 34% 69/200 0.246s >train_loss = 0.3702 train_ac